# 03 — Energy Flux Decomposition

This notebook demonstrates the full energy flux analysis in `pytrist` using `EnergyFlux` and `FieldMoments`.

## Physics background

For each particle species $s$, the total particle energy flux decomposes as:

$$Q_s = q_{\rm KE} + q_{\rm enth} + q_{\rm heat}$$

| Term | Formula | Meaning |
|------|---------|----------|
| $q_{\rm KE}$ | $\frac{1}{2}\rho_s \|\mathbf{U}_s\|^2 \cdot \mathbf{U}_s$ | Bulk kinetic energy flux |
| $q_{\rm enth}$ | $\mathbf{P}_s \cdot \mathbf{U}_s$ | Enthalpy (pressure–velocity work) |
| $q_{\rm heat}$ | $\frac{1}{2}Q_s^{\rm raw} - q_{\rm KE} - q_{\rm enth} - q_{\rm IE}$ | Irreducible heat flux cumulant |
| $q_{\rm IE}$ | $\frac{1}{2}(T_{xx}+T_{yy}+T_{zz}-\rho U^2)\,\mathbf{U}_s$ | Internal (thermal) energy flux |

plus the electromagnetic **Poynting flux** $\mathbf{S} = \frac{c}{4\pi}\mathbf{E}\times\mathbf{B}$.

**Key identity** (a useful check):
$$\frac{1}{2} Q_x^{\rm raw} = q_{\rm KE,x} + q_{\rm enth,x} + q_{\rm IE,x} + q_{\rm heat,x}$$

All quantities are normalised to $[n_0\,m_i\,v_{Ai}^3]$ in ion units.

**Before running:** set `SIM_DIR` below.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
SIM_DIR    = "/path/to/your/simulation/output"   # <── edit this
STEP       = None    # None = last step; or e.g. STEP = 10
SPECIES_ID = 2       # 1 = electrons, 2 = ions
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import pytrist

%matplotlib inline
plt.rcParams["figure.dpi"] = 120


def squeeze(arr):
    """Return 2-D x-y slice from a 3-D field array."""
    return arr[0] if arr.ndim == 3 else arr


def yavg(arr):
    """Return y-average → 1-D profile in x."""
    return squeeze(arr).mean(axis=0)

In [ ]:
sim  = pytrist.Simulation(SIM_DIR)
step = sim.steps[-1] if STEP is None else STEP
uc   = sim.unit_converter
p    = sim.params(step)

label = "electrons" if SPECIES_ID == 1 else f"ions (sp {SPECIES_ID})"
t_ion = uc.time(p.time)
print(f"Step {step}  |  t = {p.time:.1f} / ωpe = {t_ion:.3f} / Ωci")
print(f"Species: {label}")

flds = sim.fields(step)
ef   = sim.energy_flux(step)
fm   = sim.field_moments(step)

ny, nx = squeeze(flds.bz).shape
x = np.arange(nx) * uc.cell_to_di
y = np.arange(ny) * uc.cell_to_di

## 1  Spatial maps of energy flux terms

Two-dimensional colour plots of the $x$-component of each term.

In [ ]:
ke_x   = ef.bulk_ke_flux(SPECIES_ID, units="ion")["x"]
ie_x   = ef.internal_energy_flux(SPECIES_ID, units="ion")["x"]
enth_x = ef.enthalpy_flux(SPECIES_ID, units="ion")["x"]
heat_x = ef.heat_flux(SPECIES_ID, units="ion")["x"]
S_x    = ef.poynting_flux(units="ion")["x"]

panels_2d = [
    (ke_x,   r"$q_{\rm KE,x}$"),
    (ie_x,   r"$q_{\rm IE,x}$"),
    (enth_x, r"$q_{\rm enth,x}$"),
    (heat_x, r"$q_{\rm heat,x}$"),
    (S_x,    r"$S_x$"),
]

try:
    psi = squeeze(flds.psi())
except Exception:
    psi = None

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
fig.suptitle(
    rf"{label} energy flux — $t = {t_ion:.2f}\,\Omega_{{ci}}^{{-1}}$  "
    r"$[n_0\,m_i\,v_{Ai}^3]$",
    fontsize=11,
)

for ax, (data, cblabel) in zip(axes.flat, panels_2d):
    d2 = squeeze(data)
    vlim = np.percentile(np.abs(d2), 99)
    norm = mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim) if vlim > 0 else None
    im = ax.pcolormesh(x, y, d2, cmap="RdBu_r", norm=norm, shading="auto")
    fig.colorbar(im, ax=ax, label=cblabel, shrink=0.9)
    if psi is not None:
        ax.contour(x, y, psi, levels=20, colors="k", linewidths=0.4, alpha=0.5)
    ax.set_xlabel(r"$x\;[d_i]$")
    ax.set_ylabel(r"$y\;[d_i]$")
    ax.set_aspect("equal")

axes.flat[-1].set_visible(False)
plt.show()

## 2  Y-averaged profiles

Averaging over $y$ reveals the large-scale structure along $x$.

In [ ]:
q_total = ef.total_particle_energy_flux(SPECIES_ID, units="ion")

ke_xp   = yavg(ke_x)
ie_xp   = yavg(ie_x)
enth_xp = yavg(enth_x)
heat_xp = yavg(heat_x)
S_xp    = yavg(S_x)
tot_xp  = yavg(q_total["x"])

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(x, ke_xp,   label=r"$q_{\rm KE}$",   lw=1.5)
ax.plot(x, ie_xp,   label=r"$q_{\rm IE}$",   lw=1.5)
ax.plot(x, enth_xp, label=r"$q_{\rm enth}$", lw=1.5)
ax.plot(x, heat_xp, label=r"$q_{\rm heat}$", lw=1.5)
ax.plot(x, S_xp,    label=r"$S_x$",          lw=1.5)
ax.plot(x, tot_xp + S_xp, label=r"Total",    lw=2, ls="--", color="k")

ax.axhline(0, color="gray", lw=0.6, ls=":")
ax.set_xlabel(r"$x\;[d_i]$")
ax.set_ylabel(r"Flux $[n_0\,m_i\,v_{Ai}^3]$")
ax.set_title(rf"{label} energy flux profiles — $t = {t_ion:.2f}\,\Omega_{{ci}}^{{-1}}$")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## 3  Identity check: $\frac{1}{2}Q^{\rm raw} = q_{\rm KE} + q_{\rm enth} + q_{\rm IE} + q_{\rm heat}$

This identity must hold exactly by construction (modulo floating-point arithmetic).
The residual should be $\lesssim 10^{-12}$ relative to the peak flux magnitude.

In [ ]:
# total_particle_energy_flux = q_KE + q_enth + q_heat
# The left-hand side ½Q_raw = q_KE + q_enth + q_IE + q_heat = tot_prtl + q_IE
q_raw_half = tot_xp + ie_xp
lhs        = ke_xp + enth_xp + ie_xp + heat_xp
residual   = q_raw_half - lhs

peak = np.max(np.abs(lhs))
if peak > 0:
    rel_err = np.max(np.abs(residual)) / peak
    print(f"Max absolute residual: {np.max(np.abs(residual)):.3e}")
    print(f"Max relative residual: {rel_err:.3e}")
else:
    print("All fluxes are zero.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(x, q_raw_half, label=r"$\frac{1}{2}Q_x^{\rm raw}$", lw=2, color="k")
ax.plot(x, lhs,        label=r"$q_{KE}+q_{enth}+q_{IE}+q_{heat}$", lw=1.5, ls="--", color="C1")
ax.set_xlabel(r"$x\;[d_i]$")
ax.set_ylabel(r"Flux $[n_0\,m_i\,v_{Ai}^3]$")
ax.set_title(r"Identity check: LHS vs. RHS")
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(x, residual, color="C2", lw=1.5)
ax.axhline(0, color="gray", lw=0.6, ls=":")
ax.set_xlabel(r"$x\;[d_i]$")
ax.set_ylabel("Residual")
ax.set_title(r"$\frac{1}{2}Q^{\rm raw} - (q_{KE}+q_{enth}+q_{IE}+q_{heat})$")

plt.suptitle(rf"{label}  —  $t = {t_ion:.2f}\,\Omega_{{ci}}^{{-1}}$", y=1.01)
plt.tight_layout()
plt.show()

## 4  Field-file moment diagnostics

`FieldMoments` provides bulk velocity, pressure/temperature tensor, and charge density.
`sim.field_moments(step)` returns the **same** cached instance used internally by `EnergyFlux`,
so no data is read twice.

In [ ]:
vel = fm.bulk_velocity(SPECIES_ID, units="ion")
T   = fm.temperature_tensor(SPECIES_ID, units="ion")
rho = fm.charge_density(SPECIES_ID, units="ion")

print("Bulk velocity keys:", list(vel.keys()))
print("Temperature tensor keys:", list(T.keys()))
print(f"Mean |Ux| (y-avg): {yavg(vel['x']).mean():.4f} vAi")
print(f"Mean Txx  (y-avg): {yavg(T['xx']).mean():.4f} mi vAi²")

## 5  Multi-step energy budget tracking

The cell below shows how to build a time series of the peak energy flux.
It uses `clear_cache()` to avoid accumulating arrays in memory.

In [ ]:
# Uncomment and run for a full time series (may take some time for many steps)

# steps_to_use = sim.steps  # or e.g. sim.steps[::2] for every other step
# times_wci = []
# peak_enth = []
#
# for s in steps_to_use:
#     ef_s = sim.energy_flux(s)
#     p_s  = sim.params(s)
#     enth = ef_s.enthalpy_flux(SPECIES_ID, units='ion')["x"]
#     times_wci.append(uc.time(p_s.time))
#     peak_enth.append(np.percentile(np.abs(squeeze(enth)), 99))
#     ef_s.clear_cache()
#
# plt.plot(times_wci, peak_enth)
# plt.xlabel(r"$t\;[\Omega_{ci}^{-1}]$")
# plt.ylabel(r"Peak $|q_{\rm enth,x}|$  $[n_0 m_i v_{Ai}^3]$")
# plt.title("Enthalpy flux magnitude over time")
# plt.show()